# BirdCLEF 2026 — Perch v4 Track A Inference (v12)

Pipeline:
1. Load Perch v4 SavedModel
2. Run Perch on test soundscapes → embeddings + logits
3. Map logits to 234 competition species (direct + genus proxy)
4. LogReg probes for 76 species with zero Perch signal (v2 probes, full training set)
5. Class-type-aware temporal smoothing
6. File-max prior
7. Save test_perch_cache.npz for future fast-inference submissions
8. Build submission

In [ ]:
import glob
import os
import pickle
import time

import librosa
import numpy as np
import pandas as pd
import tensorflow as tf

print("TF version:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))


# ── paths ─────────────────────────────────────────────────────────────────────
# Kaggle mounts: competition data → /kaggle/input/competitions/{slug}/
#                user datasets   → /kaggle/input/datasets/{owner}/{slug}/  (no version subdir)
DATASETS = "/kaggle/input/datasets/aldisued"

COMP_DIR = "/kaggle/input/competitions/birdclef-2026"
PERCH_MODEL_DIR = f"{DATASETS}/perch-v4-model/tfhub_modules/5dcbb82658655292c50ca88ce1e6f1073b17d0d9"
PERCH_LABEL_CSV = f"{PERCH_MODEL_DIR}/assets/label.csv"
ARTIFACTS_DIR = f"{DATASETS}/birdclef2026-perch-v4-artifacts"

for name, path in [
    ("COMP_DIR", COMP_DIR),
    ("PERCH_MODEL_DIR", PERCH_MODEL_DIR),
    ("ARTIFACTS_DIR", ARTIFACTS_DIR),
]:
    exists = os.path.exists(path)
    print(f"{name}: {path}  ({'OK' if exists else 'MISSING'})")

In [ ]:
# ── load Perch v4 SavedModel ─────────────────────────────────────────────────
print("Loading Perch v4 SavedModel...")
t0 = time.time()
perch_model = tf.saved_model.load(PERCH_MODEL_DIR)
print(f"Loaded in {time.time() - t0:.1f}s")

# Quick smoke test
test_audio = tf.zeros([1, 32000 * 5], dtype=tf.float32)
logits_test, emb_test = perch_model.infer_tf(test_audio)
print(f"Smoke test — logits: {logits_test.shape}, embeddings: {emb_test.shape}")

In [ ]:
# ── load species + label mappings ────────────────────────────────────────────
taxonomy = pd.read_csv(os.path.join(COMP_DIR, "taxonomy.csv"))
competition_species = taxonomy["primary_label"].astype(str).tolist()
n_species = len(competition_species)
sp_to_idx = {sp: i for i, sp in enumerate(competition_species)}
print(f"Competition species: {n_species}")

# Class type for smoothing
TEXTURE_CLASSES = {"Amphibia", "Insecta"}  # continuous callers
label_to_class = dict(zip(taxonomy["primary_label"].astype(str), taxonomy["class_name"]))
is_texture = np.array([label_to_class.get(sp, "Aves") in TEXTURE_CLASSES for sp in competition_species])
is_event = ~is_texture
print(f"Texture species (Amphibia/Insecta): {is_texture.sum()}")
print(f"Event species (Aves + others): {is_event.sum()}")

# Load Perch label list from assets/label.csv (one eBird code per line, no header)
with open(PERCH_LABEL_CSV) as f:
    perch_labels = [line.strip() for line in f if line.strip()]
n_perch = len(perch_labels)
perch_to_idx = {lbl: i for i, lbl in enumerate(perch_labels)}
print(f"Perch v4 labels: {n_perch}")

# Direct mapping: competition species → Perch index
comp_to_perch = np.array([perch_to_idx.get(sp, -1) for sp in competition_species], dtype=np.int32)
perch_coverage = comp_to_perch >= 0
print(f"Direct Perch mapping: {perch_coverage.sum()} / {n_species}")

In [ ]:
# ── genus proxy for unmapped non-Aves species ────────────────────────────────
# For Amphibia/Insecta/Mammalia species not in Perch vocab:
# take max logit over all Perch labels sharing the same genus

# Build genus → list of Perch indices
genus_to_perch_indices = {}
for i, lbl in enumerate(perch_labels):
    # Perch labels are eBird codes (lowercase), not scientific names
    # Use first 6 chars as genus proxy for eBird codes is not reliable
    pass

# Better: use scientific name genus from taxonomy
# Perch labels are eBird codes; we need scientific names from a reference
# Since we don't have Perch's species metadata, fall back:
# unmapped species get 0.0 (Perch doesn't know them)

# Build genus mapping from taxonomy scientific names
genus_to_perch_indices = {}  # genus (lowercase) → list of Perch indices
for i, lbl in enumerate(perch_labels):
    # eBird codes: 6-char codes don't map to genus
    # Use first 3 chars as a rough genus proxy
    prefix = lbl[:3].lower()
    genus_to_perch_indices.setdefault(prefix, []).append(i)

# For competition species: extract genus from scientific_name
taxonomy["genus"] = taxonomy["scientific_name"].str.split().str[0].str.lower()

# Build per-species genus proxy index list
sp_genus_perch = {}  # sp → list of Perch indices for genus
for _, row in taxonomy.iterrows():
    sp = str(row["primary_label"])
    if comp_to_perch[sp_to_idx[sp]] >= 0:
        continue  # has direct mapping
    genus = row["genus"]
    # Try first 3 chars of genus
    prefix = genus[:3].lower() if len(genus) >= 3 else genus.lower()
    indices = genus_to_perch_indices.get(prefix, [])
    if indices:
        sp_genus_perch[sp] = indices

print(f"Unmapped species with genus proxy: {len(sp_genus_perch)} / {(~perch_coverage).sum()}")
print(f"Unmapped species without proxy: {(~perch_coverage).sum() - len(sp_genus_perch)}")

In [ ]:
# ── load LogReg probes ───────────────────────────────────────────────────────
with open(os.path.join(ARTIFACTS_DIR, "perch_probes_v2.pkl"), "rb") as f:
    probe_data = pickle.load(f)

pca = probe_data["pca"]
scaler = probe_data["scaler"]
probes = probe_data["probes"]
no_perch_species = set(probe_data["no_perch_species"])  # 76 species with zero Perch signal

n_trained = sum(1 for v in probes.values() if v is not None)
print(f"Probes loaded: {n_trained} trained probes")
print(f"No-Perch species (probe-eligible): {len(no_perch_species)}")

In [ ]:
# ── inference config ─────────────────────────────────────────────────────────
SR = 32_000
CLIP_SAMPLES = SR * 5
ALPHA_TEXTURE = 0.35  # average-neighbour smoothing for texture classes
ALPHA_EVENT = 0.15  # local-max propagation for event classes
FILE_MAX_WEIGHT = 0.05


def predict_file(y: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Run Perch on all 5s slots. Returns (slot_preds, embeddings, comp_probs)."""
    n_slots = len(y) // CLIP_SAMPLES
    if n_slots == 0:
        return (
            np.zeros((0, n_species), dtype=np.float32),
            np.zeros((0, 1280), dtype=np.float32),
            np.zeros((0, n_species), dtype=np.float32),
        )

    chunks = np.stack([y[s * CLIP_SAMPLES : (s + 1) * CLIP_SAMPLES] for s in range(n_slots)])
    x = tf.constant(chunks, dtype=tf.float32)
    raw_logits, embs = perch_model.infer_tf(x)
    raw_logits = raw_logits.numpy()
    embs = embs.numpy()  # (n_slots, 1280)

    # ── Perch probs (competition space) ──────────────────────────────────────
    comp_probs = np.zeros((n_slots, n_species), dtype=np.float32)
    comp_probs[:, perch_coverage] = 1.0 / (1.0 + np.exp(-raw_logits[:, comp_to_perch[perch_coverage]]))
    for sp, indices in sp_genus_perch.items():
        sp_idx = sp_to_idx[sp]
        max_logit = raw_logits[:, indices].max(axis=1)
        comp_probs[:, sp_idx] = 1.0 / (1.0 + np.exp(-max_logit))

    # ── Probes: only for species with zero Perch signal ──────────────────────
    embs_scaled = scaler.transform(embs)
    embs_pca = pca.transform(embs_scaled)
    out = comp_probs.copy()
    for sp_idx, sp in enumerate(competition_species):
        if sp not in no_perch_species:
            continue
        clf = probes.get(sp)
        if clf is not None:
            out[:, sp_idx] = clf.predict_proba(embs_pca)[:, 1]

    return out, embs, comp_probs


def smooth_preds(preds: np.ndarray) -> np.ndarray:
    """Vectorised class-type-aware temporal smoothing. preds: (T, n_species)"""
    if len(preds) < 2:
        return preds.copy()
    prev = np.concatenate([preds[:1], preds[:-1]])
    nxt = np.concatenate([preds[1:], preds[-1:]])
    smoothed = preds.copy()
    smoothed[:, is_texture] = (
        (1 - ALPHA_TEXTURE) * preds[:, is_texture]
        + (ALPHA_TEXTURE / 2) * prev[:, is_texture]
        + (ALPHA_TEXTURE / 2) * nxt[:, is_texture]
    )
    smoothed[:, is_event] = np.maximum(
        (1 - ALPHA_EVENT) * preds[:, is_event],
        np.maximum(
            ALPHA_EVENT * prev[:, is_event],
            ALPHA_EVENT * nxt[:, is_event],
        ),
    )
    return smoothed


print("Batched predict_file() and vectorised smooth_preds() defined.")

In [ ]:
# ── find test soundscapes ────────────────────────────────────────────────────
test_soundscape_dir = os.path.join(COMP_DIR, "test_soundscapes")
test_files = sorted(glob.glob(os.path.join(test_soundscape_dir, "*.ogg")))
print(f"Test soundscapes: {len(test_files)}")

# Load sample submission for row ordering
sample_sub = pd.read_csv(os.path.join(COMP_DIR, "sample_submission.csv"))
print(f"Sample submission rows: {len(sample_sub)}")
print(sample_sub.head(3))

In [ ]:
# ── main inference loop ──────────────────────────────────────────────────────
all_preds = {}  # row_id → pred vector (n_species,)
cache_row_ids = []  # for test cache output
cache_stems = []
cache_embeddings = []
cache_comp_probs = []
t_start = time.time()

for i, sc_path in enumerate(test_files):
    stem = os.path.splitext(os.path.basename(sc_path))[0]
    t_file = time.time()
    try:
        y, _ = librosa.load(sc_path, sr=SR, mono=True)
    except Exception as e:
        print(f"WARN: {sc_path}: {e}")
        continue

    n_slots = len(y) // CLIP_SAMPLES
    if n_slots == 0:
        continue

    slot_preds, embs, cp = predict_file(y)  # (n_slots, n_species), (n_slots, 1280), (n_slots, 234)

    # Temporal smoothing (vectorised)
    slot_preds = smooth_preds(slot_preds)

    # File-max prior
    file_max = slot_preds.max(axis=0, keepdims=True)
    slot_preds = np.clip(slot_preds + FILE_MAX_WEIGHT * file_max, 0.0, 1.0)

    for slot in range(n_slots):
        end_sec = (slot + 1) * 5
        row_id = f"{stem}_{end_sec}"
        all_preds[row_id] = slot_preds[slot]
        cache_row_ids.append(row_id)
        cache_stems.append(stem)
        cache_embeddings.append(embs[slot])
        cache_comp_probs.append(cp[slot])

    if (i + 1) % 20 == 0 or i == 0:
        elapsed = time.time() - t_start
        rate = (i + 1) / elapsed
        eta = (len(test_files) - i - 1) / max(rate, 1e-9)
        t_this = time.time() - t_file
        print(
            f"[{i + 1}/{len(test_files)}] {rate:.2f} files/s | file={t_this:.1f}s | ETA {eta / 60:.1f} min",
            flush=True,
        )

elapsed_total = time.time() - t_start
print(f"\nDone in {elapsed_total / 60:.1f} min. Rows generated: {len(all_preds)}")

# Save test cache for future fast-inference submissions
if cache_embeddings:
    np.savez_compressed(
        "test_perch_cache.npz",
        row_ids=np.array(cache_row_ids),
        stems=np.array(cache_stems),
        embeddings=np.stack(cache_embeddings).astype(np.float32),
        comp_probs=np.stack(cache_comp_probs).astype(np.float32),
        species=np.array(competition_species),
    )
    cache_mb = os.path.getsize("test_perch_cache.npz") / 1e6
    print(f"Cache saved: test_perch_cache.npz ({cache_mb:.1f} MB, {len(cache_row_ids)} slots)")

In [ ]:
# ── build submission ─────────────────────────────────────────────────────────
species_cols = [c for c in sample_sub.columns if c != "row_id"]

if all_preds:
    # Build DataFrame directly from numpy arrays — no per-species Python loop
    row_ids = list(all_preds.keys())
    preds_matrix = np.stack(list(all_preds.values()))  # (n_rows, n_species)
    sub_df = pd.DataFrame(preds_matrix, index=row_ids, columns=competition_species)
    sub_df.index.name = "row_id"
    sub_df = sub_df.reindex(sample_sub["row_id"]).fillna(0.0).reset_index()
else:
    # Sandbox: no test soundscapes injected — output sample submission as-is
    print("WARNING: no rows generated (sandbox mode), using sample submission")
    sub_df = sample_sub.copy()

sub_df = sub_df[["row_id"] + species_cols]

print(f"Submission shape: {sub_df.shape}")
print(sub_df.head(3))
sub_df.to_csv("submission.csv", index=False)
print("Saved submission.csv")